# 05 -- Textual apps

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre la philosophie Textual (TUI asyncio) ;
- creer une App Textual avec widgets et layout ;
- utiliser les widgets de base (Button, Input, Label, DataTable, ListView) ;
- gerer les evenements et bindings clavier ;
- composer avec les `Container`, `Horizontal`, `Vertical` ;
- appliquer du CSS (Textual CSS) ;
- utiliser les `reactive` attributes ;
- tester avec `App.run_test`.

## Prerequis -- ce que vous devez maitriser

- classes, heritage, decorateurs ;
- asyncio de base (async/await) ;
- notions de programmation evenementielle.

Ce que ce notebook **ne couvrira pas** :

- curses bas niveau ;
- Rich en detail (voir `04_Rich_console.ipynb`) ;
- Blessed, Urwid (alternatives).

## Plan

1. Installation et philosophie
2. Premiere app
3. Widgets de base
4. Layout : Container, Horizontal, Vertical
5. CSS Textual
6. Evenements
7. Bindings clavier
8. Reactive attributes
9. DataTable
10. Timer et intervalles
11. Screens et navigation
12. Tester une app
13. Synthese
14. Exercices

---

## 1. Installation et philosophie

In [ ]:
# pip install textual
import textual
textual.__version__ if hasattr(textual, '__version__') else 'installed'

**Textual** (Textualize) est un framework moderne pour construire des TUI (Terminal User Interfaces) avec :

- asyncio de base
- CSS pour le style
- widgets reactifs
- compatible avec Rich
- possibilite de tourner dans un navigateur (textual serve)

---

## 2. Premiere app

In [ ]:
from textual.app import App, ComposeResult
from textual.widgets import Label

class HelloApp(App):
    def compose(self) -> ComposeResult:
        yield Label('Hello Textual!')

# HelloApp().run()  # a lancer dans un terminal, pas dans Jupyter
print('App definie')

---

## 3. Widgets de base

In [ ]:
from textual.widgets import Button, Input, Label, Header, Footer, Static
print('Widgets :', [Button, Input, Label, Header, Footer, Static])

In [ ]:
from textual.app import App, ComposeResult

class FormApp(App):
    def compose(self) -> ComposeResult:
        yield Header()
        yield Label('Entrez votre nom :')
        yield Input(placeholder='Nom')
        yield Button('Valider', id='submit')
        yield Footer()

print('FormApp definie')

---

## 4. Layout : Container, Horizontal, Vertical

In [ ]:
from textual.containers import Horizontal, Vertical, Container
from textual.widgets import Button

class LayoutApp(App):
    def compose(self) -> ComposeResult:
        with Horizontal():
            yield Button('Gauche')
            yield Button('Centre')
            yield Button('Droite')
        with Vertical():
            yield Button('Haut')
            yield Button('Bas')

print('layout defini')

---

## 5. CSS Textual

In [ ]:
CSS = '''
Label {
    background: blue;
    color: white;
    padding: 1 2;
    margin: 1;
}
Button.-primary {
    background: green;
}
'''

class StyledApp(App):
    CSS = CSS
    def compose(self) -> ComposeResult:
        yield Label('Stylise')
        yield Button('OK', classes='-primary')

print('CSS inline defini')

### CSS Textual

- Selecteurs : `Label`, `#id`, `.classe`
- Proprietes : `background`, `color`, `padding`, `margin`, `width`, `height`, `border`
- Unites : `1 2` (haut-bas, gauche-droite), `100%`, `20`

---

## 6. Evenements

In [ ]:
from textual.widgets import Button

class EventApp(App):
    def compose(self) -> ComposeResult:
        yield Button('Cliquez', id='go')
        yield Label('Pret', id='status')

    def on_button_pressed(self, event: Button.Pressed) -> None:
        self.query_one('#status', Label).update(f'Clicked : {event.button.id}')

print('EventApp definie')

---

## 7. Bindings clavier

In [ ]:
from textual.binding import Binding

class KeyApp(App):
    BINDINGS = [
        Binding('q', 'quit', 'Quitter'),
        Binding('ctrl+s', 'save', 'Sauvegarder'),
        Binding('d', 'toggle_dark', 'Dark mode'),
    ]

    def action_save(self) -> None:
        self.notify('Sauvegarde !')

    def action_toggle_dark(self) -> None:
        self.dark = not self.dark

print('Raccourcis definis')

---

## 8. Reactive attributes

In [ ]:
from textual.reactive import reactive
from textual.widgets import Label

class CounterApp(App):
    count = reactive(0)

    def compose(self) -> ComposeResult:
        yield Label(id='count')
        yield Button('+', id='inc')

    def watch_count(self, value: int) -> None:
        self.query_one('#count', Label).update(f'Count: {value}')

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == 'inc':
            self.count += 1

print('Compteur reactif defini')

---

## 9. DataTable

In [ ]:
from textual.widgets import DataTable

class TableApp(App):
    def compose(self) -> ComposeResult:
        yield DataTable()

    def on_mount(self) -> None:
        table = self.query_one(DataTable)
        table.add_columns('Nom', 'Age', 'Ville')
        table.add_row('Alice', '30', 'Paris')
        table.add_row('Bob', '25', 'Lyon')
        table.add_row('Eve', '35', 'Marseille')

print('TableApp definie')

---

## 10. Timer et intervalles

In [ ]:
class ClockApp(App):
    def compose(self) -> ComposeResult:
        yield Label(id='clock')

    def on_mount(self) -> None:
        self.update_clock()
        self.set_interval(1.0, self.update_clock)

    def update_clock(self) -> None:
        import datetime
        self.query_one('#clock', Label).update(datetime.datetime.now().strftime('%H:%M:%S'))

print('Horloge definie')

---

## 11. Screens et navigation

In [ ]:
from textual.screen import Screen

class SettingsScreen(Screen):
    def compose(self) -> ComposeResult:
        yield Label('Parametres')
        yield Button('Retour', id='back')

    def on_button_pressed(self) -> None:
        self.app.pop_screen()

class MainApp(App):
    def compose(self) -> ComposeResult:
        yield Label('Ecran principal')
        yield Button('Settings', id='go-settings')

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == 'go-settings':
            self.push_screen(SettingsScreen())

print('Screens definis')

---

## 12. Tester une app

In [ ]:
# async def test_counter():
#     app = CounterApp()
#     async with app.run_test() as pilot:
#         await pilot.click('#inc')
#         assert app.count == 1
# import asyncio; asyncio.run(test_counter())
print('Pilot permet click, press, pause, wait_for_*')

---

## Synthese

| Element | Textual |
|---|---|
| App | `class MyApp(App)` |
| Composition | `def compose(self) -> ComposeResult:` |
| Widget | `yield Label('x')` |
| Layout | `with Horizontal(): ...` |
| CSS | `CSS = '...'` ou `CSS_PATH` |
| Binding | `BINDINGS = [('q', 'quit', 'Quit')]` |
| Event | `on_button_pressed(self, event)` |
| Reactive | `count = reactive(0)` + `watch_count` |
| Query | `self.query_one('#id', T)` |
| Screen | `self.push_screen(OtherScreen())` |
| Test | `async with app.run_test() as pilot:` |

### Regles a retenir

- **Textual = asyncio** : tout le runtime est async.
- **CSS** pour le style, pas dans le code.
- **Reactive** pour binding donnees -> vue.
- **`query_one`** / `query` pour acceder aux widgets.
- **`run_test` + `Pilot`** pour tester sans terminal.

---

## Exercices

### Exercice 1 (facile) -- App Hello

Creez une app qui affiche un Label avec `Bonjour Textual` et un bouton `Quitter`.

In [ ]:
from textual.app import App, ComposeResult
from textual.widgets import Label, Button

# TODO

<details>
<summary>Voir la correction</summary>

```python
from textual.app import App, ComposeResult
from textual.widgets import Label, Button

class Hello(App):
    def compose(self) -> ComposeResult:
        yield Label('Bonjour Textual')
        yield Button('Quitter', id='q')

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == 'q':
            self.exit()
```

</details>

### Exercice 2 (facile) -- Input + echo

App avec un `Input` ; quand l'utilisateur appuie sur Entree, affiche dans un Label.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from textual.app import App, ComposeResult
from textual.widgets import Input, Label

class EchoApp(App):
    def compose(self) -> ComposeResult:
        yield Input(placeholder='Tapez...')
        yield Label('', id='out')

    def on_input_submitted(self, event: Input.Submitted) -> None:
        self.query_one('#out', Label).update(f'Recu : {event.value}')
```

</details>

### Exercice 3 (moyen) -- Compteur

App avec un label affichant un compteur et deux boutons `+` et `-` qui modifient un attribut reactif.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from textual.app import App, ComposeResult
from textual.widgets import Label, Button
from textual.reactive import reactive
from textual.containers import Horizontal

class Counter(App):
    count = reactive(0)

    def compose(self) -> ComposeResult:
        yield Label('0', id='lbl')
        with Horizontal():
            yield Button('-', id='dec')
            yield Button('+', id='inc')

    def watch_count(self, v: int) -> None:
        self.query_one('#lbl', Label).update(str(v))

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == 'inc':
            self.count += 1
        elif event.button.id == 'dec':
            self.count -= 1
```

</details>

### Exercice 4 (moyen) -- Todo list

App avec un Input pour ajouter des taches et une ListView pour les afficher. Bouton `Clear` pour tout vider.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from textual.app import App, ComposeResult
from textual.widgets import Input, Button, ListView, ListItem, Label

class TodoApp(App):
    def compose(self) -> ComposeResult:
        yield Input(placeholder='Nouvelle tache...')
        yield ListView(id='list')
        yield Button('Clear', id='clear')

    def on_input_submitted(self, event: Input.Submitted) -> None:
        self.query_one('#list', ListView).append(ListItem(Label(event.value)))
        event.input.value = ''

    def on_button_pressed(self, event: Button.Pressed) -> None:
        if event.button.id == 'clear':
            self.query_one('#list', ListView).clear()
```

</details>

### Exercice 5 (difficile) -- Horloge + bindings

App avec une horloge temps reel (heure:min:sec) qui se met a jour chaque seconde, et un binding `q` pour quitter, `s` pour stopper l'horloge.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import datetime
from textual.app import App, ComposeResult
from textual.widgets import Label
from textual.binding import Binding

class Clock(App):
    BINDINGS = [
        Binding('q', 'quit', 'Quit'),
        Binding('s', 'toggle', 'Start/Stop'),
    ]

    def compose(self) -> ComposeResult:
        yield Label('00:00:00', id='clock')

    def on_mount(self) -> None:
        self.timer = self.set_interval(1.0, self.tick)

    def tick(self) -> None:
        self.query_one('#clock', Label).update(datetime.datetime.now().strftime('%H:%M:%S'))

    def action_toggle(self) -> None:
        if self.timer.running:
            self.timer.pause()
        else:
            self.timer.resume()
```

</details>

---

## Ressources externes

- [Textual documentation](https://textual.textualize.io/)
- [Textual tutorial](https://textual.textualize.io/tutorial/)
- [Textual examples](https://github.com/Textualize/textual/tree/main/examples)